## 教程

https://zhuanlan.zhihu.com/p/64702600?utm_psn=176675846615549952

https://blog.csdn.net/weixin_45417815/article/details/123678302

https://www.jianshu.com/p/1d6e0d07eb4a

## 20250327 - Pool


之前觉得pool子进程崩溃不报错 不好用 现在找到解决方法了 还是pool的代码简单。

pool有三个主要函数 map, imap, apply_async

map（f,iter），并发， 非异步

imap（f,iter），在map的基础上，异步，可以配合tqdm

apply_sync（f,arg），用这个最保险，异步，在r.get()的时候阻塞，得到错误，以后一直用下面的代码就行啦。


In [ ]:
from multiprocessing import Pool

# ✅ 任务函数
def task(x):
    if x == 3:
        raise ValueError("Simulated error")  # 模拟某个输入出错
    return x * 2

# ✅ 成功回调
def on_success(result):
    print(f"[✅ Success] Result: {result}")

# ✅ 错误回调
def on_error(e):
    print(f"[❌ Error] {e}")

if __name__ == '__main__':
    data = [1, 2, 3, 4, 5]

    with Pool(processes=4) as pool:
        results = []

        for item in data:
            res = pool.apply_async(
                task,
                args=(item,),
                callback=on_success,        # 任务成功时调用
                error_callback=on_error     # 任务出错时调用
            )
            results.append(res)

        # ✅ 确保所有任务都完成（get 会重新抛出异常）
        for r in results:
            try:
                r.get()  # 必须调用 get 才能确保任务完整执行、异常不漏掉
            except Exception as e:
                print(f"[Handled in main] {e}")

    print("🌟 All done.")

## Pool vs Process

Pool根本不报错，虽然代码比较简单，但是千万不要用，不然有错误都不知道怎么修改

Process的代码虽然比较复杂，但是容易改bug

## Process

第一种方法：
使用for p in range(cpu_count)，这种方法只会创建cpu_count个子进程，一旦进程结束，p.is_alive会变成False的状态。因此，如果使用这种方法，应该保证进程一直运行，直到任务完成，那么就需要我的func输入一个(ml_list, tbar)，让func一直从ml_list取数据，直到取完。但是这种方法使得func的参数不直观，我很少使用。

第二种方法：
创建f'{任务数}'个进程，推荐代码如下：

In [6]:
if __name__ == "__main__":
    tbar = tqdm(range(y['field']['case_num']*5))
    ps = []

    for c in range(y['field']['case_num']):  # 示例
        for s in range(1,6):
            while len(ps)>=10:
                for p in ps:
                    if not p.is_alive():
                        ps.remove(p)
            p = Process(target=func, args=(c, s))
            ps.append(p)
            p.start()
            tbar.update()

NameError: name 'tqdm' is not defined

运行时间如下：
1 cpu, 18 s
4 cpu, 45 s
6 cpu, 37 s
8 cpu, 33 s
10 cpu, 33 s
30 cpu, 33 s

## Example

In [ ]:
# feature: poro, perm, pres, sat, actions
# label: poro, perm, pres, sat, prod

import os
from pathlib import Path
import pickle
from tqdm import tqdm
from scipy.io import loadmat
from multiprocessing import Process, Manager
from yby import *

y = y_read('config.yml')
out_dir = Path(y[os.name]['out_dir'])
root_input = out_dir / 'MRST'
root_output = out_dir / 'Dataset'
actions = y_read(out_dir / 'lhs_actions.pkl')

def func(c,s):
    data_feature = {}
    data_label = {}

    file_poro_current = root_input / 'case_{:0>5}_step_{:0>2}_poro.mat'.format(c, 0)
    file_perm_current = root_input / 'case_{:0>5}_step_{:0>2}_perm.mat'.format(c, 0)
    file_pres_current = root_input / 'case_{:0>5}_step_{:0>2}_pres.mat'.format(c, s-1)
    file_sat_current = root_input / 'case_{:0>5}_step_{:0>2}_sat.mat'.format(c, s-1)

    data_feature['poro'] = Y_mrst(verbose=False).mat2df(file_poro_current, 'poro', y['grid']['cartdim'])
    data_feature['perm'] = Y_mrst(verbose=False).mat2df(file_perm_current, 'perm', y['grid']['cartdim'])
    data_feature['pres'] = Y_mrst(verbose=False).mat2df(file_pres_current, 'pres', y['grid']['cartdim'])
    data_feature['sat'] = Y_mrst(verbose=False).mat2df(file_sat_current, 'sat', y['grid']['cartdim'])
    row = actions[(actions['case']==c) & (actions['step']==s)]
    data_feature['action'] = [row.length.values[0].item(),
                              row.width.values[0].item(),
                              row.height.values[0].item(),
                              row.poro.values[0].item(),
                              row.perm.values[0].item(),
                              row.position.values[0].item()]

    file_poro_next = root_input / 'case_{:0>5}_step_{:0>2}_poro.mat'.format(c, 0)
    file_perm_next = root_input / 'case_{:0>5}_step_{:0>2}_perm.mat'.format(c, 0)
    file_pres_next = root_input / 'case_{:0>5}_step_{:0>2}_pres.mat'.format(c, s)
    file_sat_next = root_input / 'case_{:0>5}_step_{:0>2}_sat.mat'.format(c, s)
    file_prod_next = root_input / 'case_{:0>5}_step_{:0>2}_qOs.mat'.format(c, s)

    data_label['poro'] = Y_mrst(verbose=False).mat2df(file_poro_next, 'poro', y['grid']['cartdim'])
    data_label['perm'] = Y_mrst(verbose=False).mat2df(file_perm_next, 'perm', y['grid']['cartdim'])
    data_label['pres'] = Y_mrst(verbose=False).mat2df(file_pres_next, 'pres', y['grid']['cartdim'])
    data_label['sat'] = Y_mrst(verbose=False).mat2df(file_sat_next, 'sat', y['grid']['cartdim'])
    data_label['prod'] = loadmat(file_prod_next)['qOs'].item()

    # save
    dir_feature = root_output / 'feature'
    dir_feature.mkdir(parents=True, exist_ok=True)
    with open(dir_feature / 'case_{:0>5}_step_{:0>2}.pkl'.format(c,s), 'wb') as f:
        pickle.dump(data_feature, f)

    dir_label = root_output / 'label'
    dir_label.mkdir(parents=True, exist_ok=True)
    with open(dir_label / 'case_{:0>5}_step_{:0>2}.pkl'.format(c, s), 'wb') as f:
        pickle.dump(data_label, f)


if __name__ == "__main__":
    tbar = tqdm(range(y['field']['case_num']*5))
    ps = []

    for c in range(y['field']['case_num']):  # 示例
        for s in range(1,6):
            while len(ps)>=10:
                for p in ps:
                    if not p.is_alive():
                        ps.remove(p)
            p = Process(target=func, args=(c, s))
            ps.append(p)
            p.start()
            tbar.update()


## 创建多进程

### function

In [ ]:
import time
from datetime import datetime
from multiprocessing import Process

def fun(name, delay, count):
    
    print('{} start.'.format(name))
    
    while count:
        print(name, datetime.now())
        time.sleep(delay)
        count -= 1
        
    print('{} end.'.format(name))


if __name__=='__main__':
    
    for i in range(3):
        p = Process(target=fun, args=('process{}'.format(i+1), i+1, 3))
        p.start()
            
    print('main process end.')

运行结果如下，注意main process end打印的位置有问题，参考join用法。

C:\Users\yanbi\Desktop>python test.py
main process end.
process1 start.
process1 2022-11-30 19:45:40.464882
process2 start.
process2 2022-11-30 19:45:40.467873
process3 start.
process3 2022-11-30 19:45:40.468870
process1 2022-11-30 19:45:41.466941
process1 2022-11-30 19:45:42.476612
process2 2022-11-30 19:45:42.476612
process3 2022-11-30 19:45:43.473432
process1 end.
process2 2022-11-30 19:45:44.478837
process2 end.
process3 2022-11-30 19:45:46.480571
process3 end.

### class

**class.start() 会调用run方法**

**注意cla和fun在if name中写法的区别，fun需要使用Process创建类，cla已经是Process类的继承**

In [ ]:
import time
from datetime import datetime
from multiprocessing import Process

class cla(Process):
    
    def __init__(self, name, delay, count):
        super().__init__()
        
        self.name = name
        self.delay = delay
        self.count = count
        
        
    def run(self):
        
        print('{} start.'.format(self.name))
    
        while self.count:
            print(self.name, datetime.now())
            time.sleep(self.delay)
            self.count -= 1

        print('{} end.'.format(self.name))
        
        
if __name__=='__main__':
    
    for i in range(3):
        p = cla('process{}'.format(i+1), i+1, 3)
        p.start()
            
    print('main process end.')

In [ ]:
C:\Users\yanbi\Desktop>python test.py
main process end.
process1 start.
process1 2022-11-30 19:57:32.494607
process2 start.
process2 2022-11-30 19:57:32.495604
process3 start.
process3 2022-11-30 19:57:32.497599
process1 2022-11-30 19:57:33.495886
process1 2022-11-30 19:57:34.499166
process2 2022-11-30 19:57:34.499166
process1 end.
process3 2022-11-30 19:57:35.504208
process2 2022-11-30 19:57:36.508390
process3 2022-11-30 19:57:38.517761
process2 end.
process3 end.

## join

**用来阻塞进程**

### 测试1

In [3]:
import time
from datetime import datetime
from multiprocessing import Process

class cla(Process):
    
    def __init__(self, name, delay, count):
        super().__init__()
        
        self.name = name
        self.delay = delay
        self.count = count
        
        
    def run(self):
        
        print('{} start.'.format(self.name))
    
        while self.count:
            print(self.name, datetime.now())
            time.sleep(self.delay)
            self.count -= 1

        print('{} end.'.format(self.name))
        
        
if __name__=='__main__':
    
    for i in range(3):
        p = cla('process{}'.format(i+1), i+1, 3)
        p.start()
        
        p.join()
            
    print('main process end.')

main process end.


运行结果如下，发现在循环内对进程进行阻塞，并没有发挥多进程的作用。

C:\Users\yanbi\Desktop>python test.py
process1 start.
process1 2022-11-30 20:05:53.087314
process1 2022-11-30 20:05:54.103037
process1 2022-11-30 20:05:55.113568
process1 end.
process2 start.
process2 2022-11-30 20:05:56.174773
process2 2022-11-30 20:05:58.176080
process2 2022-11-30 20:06:00.182746
process2 end.
process3 start.
process3 2022-11-30 20:06:02.247957
process3 2022-11-30 20:06:05.263575
process3 2022-11-30 20:06:08.273039
process3 end.
main process end.

### 测试2

In [ ]:
import time
from datetime import datetime
from multiprocessing import Process

class cla(Process):
    
    def __init__(self, name, delay, count):
        super().__init__()
        
        self.name = name
        self.delay = delay
        self.count = count
        
        
    def run(self):
        
        print('{} start.'.format(self.name))
    
        while self.count:
            print(self.name, datetime.now())
            time.sleep(self.delay)
            self.count -= 1

        print('{} end.'.format(self.name))
        
        
if __name__=='__main__':
    
    process_list = []
    for i in range(3):
        p = cla('process{}'.format(i+1), i+1, 3)
        p.start()
        process_list.append(p)
    
    for i in process_list:
        i.join()
            
    print('main process end.')

在主进程中进行阻塞，这才是我们想要达到的目的。

C:\Users\yanbi\Desktop>python test.py
process2 start.
process2 2022-11-30 20:09:08.511166
process3 start.
process3 2022-11-30 20:09:08.512139
process1 start.
process1 2022-11-30 20:09:08.512139
process1 2022-11-30 20:09:09.514793
process1 2022-11-30 20:09:10.515477
process2 2022-11-30 20:09:10.515553
process3 2022-11-30 20:09:11.513166
process1 end.
process2 2022-11-30 20:09:12.527075
process3 2022-11-30 20:09:14.523206
process2 end.
process3 end.
main process end.

## Pipe

**用来进行进程间的资源共享，类似的方法还有queue和manager**

**Pipe在大型项目中比较快**

**queue比较灵活而且安全**

**Manager推荐使用这个**

In [ ]:
import time
from datetime import datetime
from multiprocessing import Process, Pipe

class cla1(Process):
    
    '''
    Create np.random.randint(1,10,num) array.
    '''
    
    def __init__(self, num, q1):
        super().__init__()
        
        self.num = num
        self.q1 = q1
        
        
    def run(self):
        
        import numpy as np
        
        self.data = np.random.randint(1,10, self.num)
        self.q1.send(self.data)
        print('All num: {}'.format(self.data))
        
        
class cla2(Process):
    
    '''
    Num square.
    '''
    
    def __init__(self, q2):
        super().__init__()
        
        self.q2 = q2
        
        
    def run(self):
        
        data = self.q2.recv()
        
        for i in data:
            print(i*i)
    
        
if __name__=='__main__':
    
    p1, p2 = Pipe()
    a = cla1(5, p1)
    b = cla2(p2)
    a.start()
    a.join()
    b.start()

运行结果如下：

C:\Users\yanbi\Desktop>python test.py
All num: [7 1 7 5 3]
49
1
49
25
9

## Manager

### 测试1

In [1]:
from multiprocessing import Process, Manager

def run1(data):
    data.append('run1')

def  run2(data):
    data.append('run2')

if __name__=='__main__':
    lis = Manager().list()

    p1 = Process(target=run1, args=(lis, ))
    p2 = Process(target=run2, args=(lis, ))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    print(lis)

[]


### 测试2

In [2]:
from multiprocessing import Process, Manager

class t():
    def __init__(self):
        self.run()

    def run(self):
        self.proc()

    def fun(self, manin, manout):
        while manin:
            task = manin.pop(0)
            print(task)

            manout.append(task)

    def proc(self):

        from multiprocessing import Process, Manager
        manin = Manager().list(range(10))
        manout = Manager().list()
        process = []
        for i in range(5):
            p = Process(target=self.fun, args=(manin, manout))
            process.append(p)
            p.start()

        [p.join() for p in process]

        print(manin)
        print(manout)

if __name__=='__main__':
    t()


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


## Pool

**当进程数过多时，用来限制进程数。**

**注意，args要使用元组，并且要带逗号，说明是python的args类型，否则会产生子进程打印失败的问题**

**Pool.close用来关闭进程池，使其不再接受新的任务， pool.join()用来阻塞主进程，等待进程池中的子进程结束后再关闭， close和join配合使用，顺序不能错**

### pool.apply子进程阻塞

**同样不能达到多进程的目的**

In [ ]:
import os
from multiprocessing import Process, Pool, cpu_count

def fun(num):
    
    import os
    
    print('pid: {}'.format(os.getpid()))
    print(num*num)

      
if __name__=='__main__':
    
    print('main process pid: {}'.format(os.getpid()))
    print('This PC has {} logical cores.'.format(cpu_count()))
    pool = Pool(3)
    for i in range(6):
        pool.apply(func=fun, args=(i,))
        
    pool.close()
    pool.join()

运行结果如下：

C:\Users\yanbi\Desktop>python test.py
main process pid: 17560
This PC has 16 logical cores.
pid: 10396
0
pid: 10396
1
pid: 14724
4
pid: 10396
9
pid: 14724
16
pid: 10392
25

### pllo.apply_async子进程不阻塞

In [ ]:
import os
from multiprocessing import Process, Pool, cpu_count

def fun(num):
    
    import os
    
    print('pid: {}'.format(os.getpid()))
    print(num*num)

      
if __name__=='__main__':
    
    print('main process pid: {}'.format(os.getpid()))
    print('This PC has {} logical cores.'.format(cpu_count()))
    pool = Pool(3)
    for i in range(6):
        pool.apply_async(func=fun, args=(i,))
        
    pool.close()
    pool.join()

运行结果如下：

C:\Users\yanbi\Desktop>python test.py
main process pid: 2968
This PC has 16 logical cores.
pid: 12012
0
pid: 12012
1
pid: 12012
4
pid: 12012
9
pid: 6120
pid: 12012
16
25

## Manager

In [1]:
from multiprocessing import Process, Manager

def run1(data):
    data.append('run1')

def  run2(data):
    data.append('run2')

if __name__=='__main__':
    lis = Manager().list()

    p1 = Process(target=run1, args=(lis, ))
    p2 = Process(target=run2, args=(lis, ))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    print(lis)

[]


## 笔记

**jupyter不能用来测试多进程**

### 进程vs线程

进程是python中最小的资源分配单位，各个进程之间的数据和内存是不共享的。

线程是操作系统能够运行调度的最小单位，包含在进程之中，各个子线程共享所在进程的内存数据。

一个进程可以包含多个并发线程，一个进程至少要包含一个线程（主线程），线程与线程之间没有隶属关系，进程只能操作子进程。

并发是在一个进程内多线程抢占资源的，因此提高资源利用率，要创建多进程，而不是多线程。

### 进程间的通信

主要有3种通信方式，分别是quene，pipes，managers

quene的命令比较丰富和安全（更容易操作）。
pipes被认为是最快的方式。

### 守护进程

守护进程会在主进程代码执行结束后就终止，守护进程内无法再开启子进程。

### 进程锁

进程之间数据不共享，为了保证各个进程访问共享数据时的数据安全，加锁处理可以避免访问的冲突。

串行：一个时间点只能执行一个任务，执行完了才能执行下一个任务。

并行：当系统有一个以上cpu（指核心，非物理）时，当一个cpu核心执行一个进程时，另一个cpu核心可以执行另一个进程，两个进程互不抢占cpu资源。

并发：在一个进程内，多线程并发，各个线程抢占资源。

### 父进程与子进程

在windows中使用多进程，需要添加if __name__=='__main__'，这样系统才会判断出这个父进程，避免递归创建子进程。

而在linux中，系统会fork出分支进程，主进程返回0，子进程返回进程编号，这样也可以达到识别父子进程的目的，所以linux不用加。